Anomaly Detection with Graph Attention Networks

In [72]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from tqdm.auto import tqdm
from torch_geometric.utils import add_self_loops, softmax
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


torch.manual_seed(42)
np.random.seed(42)

engine_faultdb = 'EngineFaultDB/EngineFaultDB_Final.csv'
df = pd.read_csv(engine_faultdb)

In [73]:
from torch.utils.data import Dataset 
from torch_geometric.data import Data

class EngineGraphDataset(Dataset):
    def __init__(self, X, y, edge_index, num_features):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.edge_index = edge_index
        self.num_features = num_features
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # Create a graph for this sample
        x = self.X[idx].unsqueeze(1)  # [num_features, 1]
        
        return Data(
            x=x,
            edge_index=self.edge_index,
            y=self.y[idx]
)

In [74]:
class GraphLayer(MessagePassing):
    """
    Graph Attention layer as described in GDN paper
    Incorporates sensor embeddings into attention mechanism
    """
    def __init__(self, in_dim, out_dim, embed_dim):
        super(GraphLayer, self).__init__(aggr='add')
        
        # Feature transformation
        self.lin = nn.Linear(in_dim, out_dim)
        
        self.att = nn.Linear(2 * (embed_dim + out_dim), 1)
        
        self.embed_dim = embed_dim
        self.out_dim = out_dim
    
    def forward(self, x, edge_index, embeddings, return_attention=False):
        """
        x: Node features [num_nodes, in_dim]
        edge_index: Graph edges [2, num_edges]
        embeddings: Sensor embeddings [num_nodes, embed_dim]
        """
        
        x_transformed = self.lin(x)
        
        x_with_embed = torch.cat([embeddings, x_transformed], dim=1)
        
        # Pass return_attention to propagate
        out = self.propagate(
            edge_index, 
            x=x_with_embed,
            return_attention=return_attention
        )
        
        return out
    
    def message(self, x_i, x_j, edge_index_i, return_attention=False):
        """
        x_i: Target node features [num_edges, embed_dim + out_dim]
        x_j: Source node features [num_edges, embed_dim + out_dim]
        """
        # Concatenate source and target features for attention
        x_cat = torch.cat([x_i, x_j], dim=-1)
        
        alpha = self.att(x_cat)
        alpha = F.leaky_relu(alpha, negative_slope=0.2)
        
        alpha = softmax(alpha, edge_index_i)
        
        if return_attention:
            self._alpha = alpha
        
        return alpha * x_j[:, -self.out_dim:]  # Only use transformed features for message
    
    def update(self, aggr_out):
        return F.relu(aggr_out)

In [75]:
# %%
from torch_geometric.data import Data, Batch

class GDN(nn.Module):
    """
    GDN using PyG's native batching mechanism
    """
    def __init__(self, num_features, hidden_dim, output_dim, 
                 embed_dim=64, topk=20):
        super(GDN, self).__init__()
        
        self.num_features = num_features
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.embed_dim = embed_dim
        self.topk = min(topk, num_features - 1)
        
        self.embeddings = nn.Parameter(torch.randn(num_features, embed_dim))
        nn.init.xavier_uniform_(self.embeddings)
        
        self.graph_layer1 = GraphLayer(1, hidden_dim, embed_dim)
        self.graph_layer2 = GraphLayer(hidden_dim, hidden_dim, embed_dim)
        
        # Global pooling and classification
        from torch_geometric.nn import global_mean_pool
        self.pool = global_mean_pool
        
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(0.3)
        self.register_buffer('edge_index', None)
    
    def learn_graph_structure(self):
        """Learn graph structure"""
        
        embeddings_norm = F.normalize(self.embeddings, p=2, dim=1)
        similarity = torch.mm(embeddings_norm, embeddings_norm.t())
        similarity.fill_diagonal_(-1e9)
        
        _, topk_indices = torch.topk(similarity, self.topk, dim=1)
        
        edge_list = []
        for i in range(self.num_features):
            for j in topk_indices[i]:
                edge_list.append([j.item(), i])
        
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
        
        return edge_index
    
    def forward(self, batch_data):
        """
        batch_data: PyG Batch object containing multiple graphs
        """
        if self.edge_index is None:
            self.edge_index = self.learn_graph_structure().to(batch_data.x.device)
        
        # Get batch-specific embeddings
        num_graphs = batch_data.batch.max().item() + 1
        batch_embeddings = self.embeddings.repeat(num_graphs, 1)
        
        # Graph layers
        h1 = self.graph_layer1(batch_data.x, batch_data.edge_index, batch_embeddings)
        h1 = self.dropout(h1)
        
        h2 = self.graph_layer2(h1, batch_data.edge_index, batch_embeddings)
        h2 = h2 * batch_embeddings[:, :self.hidden_dim]
        
        # Global pooling
        h_pooled = self.pool(h2, batch_data.batch)
        
        # Classification
        h = F.relu(self.fc1(h_pooled))
        h = self.dropout(h)
        out = self.fc2(h)
        
        return F.log_softmax(out, dim=1)


In [76]:
train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42, 
    shuffle=True, stratify=df['Fault']
)

y_train = train_data['Fault'].values.astype(int)
X_train = train_data.drop(columns=['Fault'])

y_test = test_data['Fault'].values.astype(int)
X_test = test_data.drop(columns=['Fault'])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.LongTensor(y_test)

In [ ]:
from torch_geometric.loader import DataLoader 

num_features = X_train_scaled.shape[1]
num_classes = len(np.unique(y_train))

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

print(f"Using device: {device}")


# dummy model to get edge_index
temp_model = GDN(num_features, 64, num_classes, 64, 15)
edge_index = temp_model.learn_graph_structure()

train_dataset = EngineGraphDataset(X_train_scaled, y_train, edge_index, num_features)
test_dataset = EngineGraphDataset(X_test_scaled, y_test, edge_index, num_features)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model = GDN(num_features, 64, num_classes, 64, 15).to(device)

model = GDN(
    num_features=num_features,
    hidden_dim=64,
    output_dim=num_classes,
    embed_dim=64,
    topk=30  # Each feature connected to 30 most similar features (increased from 15 since rpm <-> speed and speed<->rpm counts as a single feature)
).to(device)

Using device: mps


In [78]:
def evaluate(model, loader, device, desc='Evaluating'):
    """Evaluate model with PyG DataLoader"""
    model.eval()
    y_true = []
    y_pred = []
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=desc, leave=False)
    
    with torch.no_grad():
        for batch_data in pbar:
            batch_data = batch_data.to(device)
            output = model(batch_data)
            pred = output.argmax(dim=1)
            
            y_true.extend(batch_data.y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())
            
            correct += (pred == batch_data.y).sum().item()
            total += batch_data.num_graphs
            
            pbar.set_postfix({'acc': f'{100.0 * correct / total:.2f}%'})
    
    accuracy = accuracy_score(y_true, y_pred)
    return accuracy, y_true, y_pred

In [79]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
criterion = nn.NLLLoss()

epochs = 100
best_test_acc = 0

print("\nTraining GDN...")
print("="*60)

for epoch in tqdm(range(epochs), desc='Training Progress'):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_data in train_loader:
        batch_data = batch_data.to(device)
        
        optimizer.zero_grad()
        output = model(batch_data)
        loss = criterion(output, batch_data.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_data.num_graphs
        pred = output.argmax(dim=1)
        correct += (pred == batch_data.y).sum().item()
        total += batch_data.num_graphs
    
    train_loss = total_loss / total
    train_acc = correct / total
    
    if epoch % 10 == 0:
        test_acc, _, _ = evaluate(model, test_loader, device, desc='')
        
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save(model.state_dict(), 'best_gdn_tabular_model.pt')
        
        tqdm.write(f'Epoch {epoch:03d}: Loss={train_loss:.4f}, ' f'Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}, 'f'Best={best_test_acc:.4f}')

print("="*60)
print(f"Best test accuracy: {best_test_acc:.4f}")



Training GDN...


Training Progress:   1%|          | 1/100 [00:20<33:05, 20.05s/it]

Epoch 000: Loss=1.2673, Train Acc=0.3583, Test Acc=0.4292, Best=0.4292


Training Progress:  11%|█         | 11/100 [02:55<23:52, 16.10s/it]

Epoch 010: Loss=0.7544, Train Acc=0.6149, Test Acc=0.6421, Best=0.6421


...

Epoch 080: Loss=0.4491, Train Acc=0.7266, Test Acc=0.7433, Best=0.7433


Training Progress:  91%|█████████ | 91/100 [39:08<02:29, 16.66s/it]

Epoch 090: Loss=0.4422, Train Acc=0.7269, Test Acc=0.7416, Best=0.7433


Training Progress: 100%|██████████| 100/100 [41:56<00:00, 25.16s/it]

Best test accuracy: 0.7433


In [80]:
model.load_state_dict(torch.load('best_gdn_tabular_model.pt'))

embeddings = model.embeddings.detach().cpu().numpy()

print(f"\n{'='*60}")
print("(OPTIONAL): FEATURE EMBEDDINGS ANALYSIS")
print(f"{'='*60}")

embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print("Computing pairwise similarities...")
similarity_matrix = np.zeros((num_features, num_features))

for i in tqdm(range(num_features), desc='Computing similarities'):
    similarity_matrix[i] = np.dot(embeddings_norm[i], embeddings_norm.T)

np.fill_diagonal(similarity_matrix, 0)

print("\nFinding most similar feature pairs...")
similarity_flat = similarity_matrix.flatten()
top_indices = np.argsort(similarity_flat)[-30:][::-1]
top_pairs = [(idx // num_features, idx % num_features) for idx in top_indices]

print("\nTop 10 Most Similar Feature Pairs:")
for i, (f1, f2) in enumerate(top_pairs, 1):
    sim = similarity_matrix[f1, f2]
    print(f"{i}. {X_train.columns[f1]:30s} <-> {X_train.columns[f2]:30s}: {sim:.4f}")


(OPTIONAL): FEATURE EMBEDDINGS ANALYSIS
Computing pairwise similarities...


Computing similarities: 100%|██████████| 14/14 [00:00<00:00, 7773.40it/s]


Finding most similar feature pairs...

Top 10 Most Similar Feature Pairs:
1. RPM                            <-> Speed                         : 0.9912
2. Speed                          <-> RPM                           : 0.9912
3. HC                             <-> AFR                           : 0.9754
4. AFR                            <-> HC                            : 0.9754
5. AFR                            <-> CO                            : 0.9750
6. CO                             <-> AFR                           : 0.9750
7. HC                             <-> CO                            : 0.9613
8. CO                             <-> HC                            : 0.9613
9. TPS                            <-> Power                         : 0.9596
10. Power                          <-> TPS                           : 0.9596
11. Force                          <-> Power                         : 0.9276
12. Power                          <-> Force                         : 0.927

In [81]:
# %%
# Load best model and evaluate
model.load_state_dict(torch.load('best_gdn_tabular_model.pt'))
test_acc, y_true, y_pred = evaluate(model, test_loader, device, desc='Test')

print(f"\nTest Accuracy: {test_acc:.4f}\n")
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("Classification Report:")
print(classification_report(y_true, y_pred, digits=4, target_names=["Fault0", "Fault1", "Fault2", "Fault3"]))



Test Accuracy: 0.7433


Confusion Matrix:
[[3169    4   12   15]
 [  27 2159    4   10]
 [  34    0 2253  713]
 [  40    1 2015  744]]
Classification Report:
              precision    recall  f1-score   support

      Fault0     0.9691    0.9903    0.9796      3200
      Fault1     0.9977    0.9814    0.9895      2200
      Fault2     0.5259    0.7510    0.6186      3000
      Fault3     0.5020    0.2657    0.3475      2800

    accuracy                         0.7433     11200
   macro avg     0.7487    0.7471    0.7338     11200
weighted avg     0.7392    0.7433    0.7268     11200

